# 第 02 章：把自然语言计划变成业务契约（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch02-model-structured-output`

In [3]:
from langchain_core.messages import AIMessage

from mini_deerflow.models import create_offline_model
from mini_deerflow.schemas import ResearchRequest

structured_model = create_offline_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "ResearchRequest",
                    "args": {
                        "question": "LangGraph 如何恢复长任务？",
                        "deliverable": "带引用的中文说明",
                        "max_sources": 4,
                    },
                    "id": "structured-1",
                    "type": "tool_call",
                }
            ],
        )
    ]
).with_structured_output(ResearchRequest)
research_request = structured_model.invoke("把用户需求整理成研究请求")
assert isinstance(research_request, ResearchRequest)
assert research_request.max_sources == 4
research_request


### 实验 `ch02-task-plan`

In [1]:
from mini_deerflow.schemas import PlanStep, TaskPlan

plan = TaskPlan(
    objective="解释 LangGraph durable execution",
    steps=[
        PlanStep(id="research", instruction="检索官方持久化资料"),
        PlanStep(
            id="write",
            instruction="生成带引用的中文说明",
            depends_on=["research"],
        ),
    ],
)
assert plan.schema_version == 1
assert plan.steps[1].depends_on == ["research"]
plan.model_dump()


{'schema_version': 1,
 'objective': '解释 LangGraph durable execution',
 'steps': [{'id': 'research', 'instruction': '检索官方持久化资料', 'depends_on': []},
  {'id': 'write', 'instruction': '生成带引用的中文说明', 'depends_on': ['research']}]}

### 实验 `ch02-outcome-triad`

In [2]:
from mini_deerflow.schemas import StructuredFailure, validate_research_request

refusal = StructuredFailure.refused("请求涉及未授权数据")
invalid = validate_research_request(
    {"question": "", "deliverable": "报告", "max_sources": 0}
)
assert refusal.kind == "refusal"
assert isinstance(invalid, StructuredFailure)
assert invalid.kind == "validation_error"


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch02-artifact-boundary`

In [6]:
from pydantic import ValidationError

from mini_deerflow.schemas import ArtifactRef

try:
    ArtifactRef(path="../secret.txt", media_type="text/plain")
except ValidationError as error:
    artifact_error = error
else:
    raise AssertionError("越界路径必须被拒绝")

assert "工作区内的相对路径" in str(artifact_error)


### 实验 `ch02-subagent-failure`

In [7]:
from mini_deerflow.schemas import SubagentResult

failed = SubagentResult.failed("research", "timeout")
assert failed.status == "failed"
assert failed.error == "timeout"
assert failed.artifacts == []
failed.model_dump()


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。